## Rule analysis and correction mechanism

It is assumed that when comparing the prices of Limited and Casco models one must do so only between the same variant and the same deductible amount. If this assumption is not made, for instance, we could compare casco_compact_500 (price: 620) with limited_casco_premium_100 (price: 1100). In this situation the fact that 620 is less than 1100 would be treated as a violation, even though there is no basis for doing so in reality. MTPL, on the other hand, has no variants or deductibles, but rather a single base price which is then compared with the lowest price obtained at the higher level.

The results indicate that 11 violations were identified among the 24 product-level comparisons. More precisely, in a group of 12 comparisons involving the Limited and Casco packages, 11 violations were found, whereas all 12 of the comparisons between the MTPL and the Limited are correct. The violations all show the same trend, namely that Casco is cheaper than Limited. This uniform pattern suggests that the problem lies in a systemic miscalibration of the entire Casco block and not in the 11 separate anomalies.

The problem is that the Casco policy, being cheaper than the Limited one, gives the customer greater coverage at a lower price. This may steer customers towards Casco while the insurer takes on the additional own-damage risk without a corresponding price increase.

For this example, I chose to increase the prices of the Casco block. Reducing the prices of the Limited products would require additional evidence from claims costs, margins, loss history or competitive pricing, none of which is available in the input.

Using the working 50% threshold, 91.7% of the 12 comparisons fail, so a full block shift is applied rather than individual local corrections.

The same pre-rounding multiplicative factor is applied to the whole block. It is calculated from the highest (Limited + 1) / Casco ratio, which is 1.09467; the simple Limited/Casco ratio of 1.0933 is not enough since 750 multiplied by 1.0933 would round up to exactly 820 and the comparison needs a strict inequality. Individual adjustments would distort the internal price relationships. For instance, the deductible step for casco_compact would change from -17.3% to -20.7%. With a common factor it remains approximately -17.3%, with only small rounding effects.

A drawback of this method is that if there were an extreme or unusual value at the lowest level, namely MTPL, it would be passed up through both blocks and as a result Casco would end up 28% higher than its original price; in practice, this mechanism would need an upper limit on the maximum permitted percentage change.


In [1]:
example_prices_to_correct = {
    "mtpl": 400,
    "limited_casco_compact_100": 820,
    "limited_casco_compact_200": 760,
    "limited_casco_compact_500": 650,
    "limited_casco_basic_100": 900,
    "limited_casco_basic_200": 780,
    "limited_casco_basic_500": 600,
    "limited_casco_comfort_100": 950,
    "limited_casco_comfort_200": 870,
    "limited_casco_comfort_500": 720,
    "limited_casco_premium_100": 1100,
    "limited_casco_premium_200": 980,
    "limited_casco_premium_500": 800,
    "casco_compact_100": 750,
    "casco_compact_200": 700,
    "casco_compact_500": 620,
    "casco_basic_100": 830,
    "casco_basic_200": 760,
    "casco_basic_500": 650,
    "casco_comfort_100": 900,
    "casco_comfort_200": 820,
    "casco_comfort_500": 720,
    "casco_premium_100": 1050,
    "casco_premium_200": 950,
    "casco_premium_500": 780,
}

In [2]:
import math


def pricing_enforcement(prices: dict[str, int]) -> dict[str, int]:
    products = ["limited_casco", "casco"]
    variants = ["compact", "basic", "comfort", "premium"]
    deductibles = [100, 200, 500]
    descending = deductibles[::-1]
    # The order prevents the structure from being broken. First, the internal relations (deductible and variant) are adjusted, and then the product levels are processed (MTPL before Casco).
    rule_order = ["deductible", "variant", "product_level_mtpl", "product_level_casco"]
    higher_block = {"product_level_mtpl": "limited_casco", "product_level_casco": "casco"}
    constraints = []
    for product in products:
        for variant in variants:
            for cheaper_deductible, dearer_deductible in zip(descending, descending[1:]):
                constraints.append((f"{product}_{variant}_{cheaper_deductible}", f"{product}_{variant}_{dearer_deductible}", "deductible"))
        for deductible in deductibles:
            for entry in ["compact", "basic"]:
                constraints.append((f"{product}_{entry}_{deductible}", f"{product}_comfort_{deductible}", "variant"))
            constraints.append((f"{product}_comfort_{deductible}", f"{product}_premium_{deductible}", "variant"))
    for variant in variants:
        for deductible in deductibles:
            constraints.append(("mtpl", f"limited_casco_{variant}_{deductible}", "product_level_mtpl"))
            constraints.append((f"limited_casco_{variant}_{deductible}", f"casco_{variant}_{deductible}", "product_level_casco"))
    corrected = dict(prices)
    print(f"violations before: {sum(corrected[cheap] >= corrected[dear] for cheap, dear, _ in constraints)}")
    for _ in range(10):
        if not any(corrected[cheap] >= corrected[dear] for cheap, dear, _ in constraints):
            break
        for rule in rule_order:
            relevant = [triple for triple in constraints if triple[2] == rule]
            violated = [triple for triple in relevant if corrected[triple[0]] >= corrected[triple[1]]]
            if not violated:
                continue
            block = higher_block.get(rule)
            internal_valid = block is not None and all(
                corrected[cheap] < corrected[dear]
                for cheap, dear, other_rule in constraints
                if other_rule in ("deductible", "variant") and dear.startswith(block)
            )
            # A threshold of 50% separates a systemic block error from an individual anomaly. A block is moved only if the internal relations in it are already correct.
            if internal_valid and 2 * len(violated) >= len(relevant):
                # +1 is added to the numerator due to strict inequality. The standard ratio after rounding (750 * 1.0933) would give exactly 820, so the violation would remain.
                factor = max((corrected[cheap] + 1) / corrected[dear] for cheap, dear, _ in violated)
                print(f"{rule}: {len(violated)}/{len(relevant)} violated, block {block} x {factor:.6f}")
                for key in [f"{block}_{variant}_{deductible}" for variant in variants for deductible in deductibles]:
                    previous = corrected[key]
                    # Prices are integers; ceil keeps an uplift from rounding a corrected price back to equality.
                    corrected[key] = math.ceil(previous * factor)
                    print(f"  {key}: {previous} -> {corrected[key]}")
            else:
                print(f"{rule}: {len(violated)}/{len(relevant)} violated, local")
                for cheap, dear, _ in violated:
                    previous = corrected[dear]
                    corrected[dear] = corrected[cheap] + 1
                    print(f"  {dear}: {previous} -> {corrected[dear]}")
    remaining = sum(corrected[cheap] >= corrected[dear] for cheap, dear, _ in constraints)
    assert remaining == 0, "correction did not converge"
    print(f"violations after: {remaining}")
    return corrected


In [3]:
corrected_prices = pricing_enforcement(example_prices_to_correct)

violations before: 11
product_level_casco: 11/12 violated, block casco x 1.094667
  casco_compact_100: 750 -> 821
  casco_compact_200: 700 -> 767
  casco_compact_500: 620 -> 679
  casco_basic_100: 830 -> 909
  casco_basic_200: 760 -> 832
  casco_basic_500: 650 -> 712
  casco_comfort_100: 900 -> 986
  casco_comfort_200: 820 -> 898
  casco_comfort_500: 720 -> 789
  casco_premium_100: 1050 -> 1150
  casco_premium_200: 950 -> 1040
  casco_premium_500: 780 -> 854
violations after: 0


In [4]:
print(pricing_enforcement(corrected_prices) == corrected_prices)

violations before: 0
violations after: 0
True


In [5]:
single_anomaly_prices = {
    "mtpl": 400,
    "limited_casco_compact_100": 800,
    "limited_casco_compact_200": 720,
    "limited_casco_compact_500": 650,
    "limited_casco_basic_100": 820,
    "limited_casco_basic_200": 740,
    "limited_casco_basic_500": 660,
    "limited_casco_comfort_100": 900,
    "limited_casco_comfort_200": 810,
    "limited_casco_comfort_500": 730,
    "limited_casco_premium_100": 1000,
    "limited_casco_premium_200": 900,
    "limited_casco_premium_500": 810,
    "casco_compact_100": 920,
    "casco_compact_200": 830,
    "casco_compact_500": 640,
    "casco_basic_100": 940,
    "casco_basic_200": 850,
    "casco_basic_500": 760,
    "casco_comfort_100": 1030,
    "casco_comfort_200": 930,
    "casco_comfort_500": 840,
    "casco_premium_100": 1150,
    "casco_premium_200": 1030,
    "casco_premium_500": 930,
}

single_anomaly_corrected = pricing_enforcement(single_anomaly_prices)

violations before: 1
product_level_casco: 1/12 violated, local
  casco_compact_500: 640 -> 651
violations after: 0
